# The last two feature experiments, and freezing the model

The Day 6 brief says no new features after this point, so this is where the model gets
its final shape. Two fixes were tried, each aimed at a problem found by checking rather
than guessed at:

1. **The China gap.** Player rating was never recorded for all of China Kickoff and much of
   China Stage 1, so on those maps the model is told "no difference between the teams" when
   it actually knows nothing. Combat score *was* recorded, and could stand in.
2. **Thin records.** A player judged on three maps can look brilliant or awful purely by luck,
   and the model believes that just as much as a record built on thirty. Pulling thin records
   toward the average player is the standard fix.

**Result:** the China fix is kept, thin-record shrinkage is dropped. The model is now
logistic regression on two columns — rating difference (combat score where rating is
missing) and who picked the map — and it is frozen.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

sys.path.insert(0, str(Path.cwd().parent / "src"))

pd.set_option("display.width", 180)

import form as form_module
from dataset import build_map_table, season_folds, split_season
from features import build_features, feature_columns
from model import FINAL_COLUMNS, make_final_model, make_model, picker_baseline

HALF_LIFE = 5

maps = build_map_table()
learn, test = split_season(maps)
X_agents, y = build_features(learn, feature_columns(maps))
form = form_module.player_form(learn, half_life=HALF_LIFE)
X = pd.concat([X_agents, form], axis=1)
baseline = picker_baseline(y, learn)

print(f"learning half: {len(learn):,} maps   (test half of {len(test):,} not touched)")

learning half: 756 maps   (test half of 516 not touched)


## 1. How big is the China gap?

A side's rating form is "unknown" only when none of its five players has a single recorded
rating yet. Some of those maps can be rescued by combat score; some cannot, because nobody
on the side has played at all yet — the opening matches of each Kickoff.

In [2]:
gaps = learn[["event"]].assign(
    rating_unknown=form["rating_diff"].isna(),
    acs_unknown=form["acs_diff"].isna(),
)
recoverable = gaps["rating_unknown"] & ~gaps["acs_unknown"]

print(f"maps with rating form unknown      : {gaps['rating_unknown'].sum()}")
print(f"  of which combat score can rescue : {recoverable.sum()}")
print(f"  unknown either way (nobody has played yet): {gaps['acs_unknown'].sum()}")
print()
by_event = gaps.groupby("event")[["rating_unknown", "acs_unknown"]].sum()
by_event[by_event["rating_unknown"] > 0]

maps with rating form unknown      : 160
  of which combat score can rescue : 79
  unknown either way (nobody has played yet): 81



,rating_unknown,acs_unknown
event,,
VCT 2025 Americas Kickoff,17,17
VCT 2025 China Kickoff,56,19
VCT 2025 China Stage 1,38,0
VCT 2025 EMEA Kickoff,22,22
VCT 2025 Pacific Kickoff,23,23
Valorant Masters Bangkok 2025,4,0


79 maps — about one in ten — can be rescued, almost all of them Chinese. That caps what
this fix can do for accuracy at roughly a point. The stronger reason for it is coverage: at
the moment the model knows nothing about one region for a third of the learning half.

## 2. Does the model know when it's right?

A useful model should be right more often when it is confident. This checks the model as it
stood before these experiments.

In [3]:
def predictions(pipeline, columns):
    """Every validation prediction across the five folds."""
    Xf = X[columns].astype(float)
    out = []
    for learn_rows, check_rows in season_folds(learn):
        model = clone(pipeline).fit(Xf.loc[learn_rows], y.loc[learn_rows])
        chance = model.predict_proba(Xf.loc[check_rows])[:, 1]
        out.append(pd.DataFrame({"chance": chance, "truth": y.loc[check_rows].to_numpy()},
                                index=check_rows))
    p = pd.concat(out)
    p["confidence"] = np.maximum(p["chance"], 1 - p["chance"])
    p["right"] = (p["chance"] >= 0.5) == p["truth"]
    return p


before = predictions(make_model(LogisticRegression(max_iter=5000)), ["rating_diff", "map_picked_by_a"])
before["band"] = pd.cut(before["confidence"], [0.5, 0.55, 0.6, 0.65, 0.7, 1.0], include_lowest=True,
                        labels=["50-55%", "55-60%", "60-65%", "65-70%", "70%+"])

(before.groupby("band", observed=True)
 .agg(maps=("right", "size"), actually_right=("right", "mean"))
 .assign(actually_right=lambda t: (t["actually_right"] * 100).round(1)))

,maps,actually_right
band,,
50-55%,234,59.8
55-60%,240,60.8
60-65%,92,53.3
65-70%,45,68.9
70%+,18,55.6


**No.** When the model says 50–55% it is right about 60% of the time; when it says 70%+ it
is right about 56%. Its confidence tells you almost nothing. It has learned *which* side has
been playing better, but not *how much* better.

One likely reason: how much history sits behind each prediction.

In [4]:
# How many rated maps the LESS-known side had behind it, going into each map.
players = form_module.load_player_maps(learn).sort_values(["played_at", "match_id"])
rated, history = {}, {}
for (_, match_id), match in players.groupby(["played_at", "match_id"], sort=True):
    team_a, team_b = match.iloc[0]["team_a"], match.iloc[0]["team_b"]

    def side(team):
        return np.mean([rated.get(n, 0) for n in match[match["Team"] == team]["Player"].unique()])

    history[match_id] = min(side(team_a), side(team_b))
    for _, performance in match.iterrows():
        if pd.notna(performance["Rating"]):
            rated[performance["Player"]] = rated.get(performance["Player"], 0) + 1

before["history"] = learn.loc[before.index, "match_id"].map(history)
before["history_band"] = pd.cut(before["history"], [-1, 5, 10, 20, 1000],
                                labels=["under 5 maps", "5-10", "10-20", "20+"])

(before.groupby("history_band", observed=True)
 .agg(maps=("right", "size"), right=("right", "mean"))
 .assign(right=lambda t: (t["right"] * 100).round(1)))

,maps,right
history_band,,
under 5 maps,150,58.0
5-10,155,58.1
10-20,200,59.5
20+,124,64.5


With 20+ maps of history behind both sides the model gets about 64% right; with thin
records, about 58%. Its most confident calls also tend to rest on *less* history, not more —
the signature of believing extreme numbers from small samples.

One caution: 20+ maps of history only exists later in the season, when the model also has
more to learn from. So this is suggestive, not proof, and 124 maps is a small group.

## 3. The rules, set before running anything

Every extra thing tried is another chance to find a winner by luck, and the margin of error
here is ±4 to 6 points. So the rules for keeping a change were fixed first:

- Compared with the current model, it must be **better on at least 3 of the 5 folds, worse
  on at most 1**, and its **log loss must be no worse**.
- Shrinkage is tried at **three strengths only — 3, 5 and 10 maps**. If more than one passes,
  5 is used, chosen in advance as the middle value, not whichever scores highest.

A strength of 5 means a player who has played 5 maps is judged halfway between their own
record and the average player.

## 4. The two fixes

**China fix.** Where a side's rating form is unknown, its combat-score difference is used
instead, converted to the rating scale by a single number worked out from each fold's
training rows. It lives in `src/model.py` as `RatingFallback`, part of `make_final_model()`.

**Shrinkage.** Defined below rather than in `src/`, because it was dropped. It walks through
the season exactly as `src/form.py` does, with one extra step: each player's form is pulled
toward the average player by an amount that shrinks as they play more maps. The average is
worked out from the first chunk of the season only, which every fold trains on, so no
fold's checking maps can influence it.

In [5]:
STATS = {"Rating": "rating", "Average Combat Score": "acs"}

first_rows, _ = next(season_folds(learn))
first_players = form_module.load_player_maps(learn.loc[first_rows])
AVERAGE_PLAYER = {column: float(first_players[column].mean()) for column in STATS}


def form_shrunk(k):
    """shrunk = average + (their form - average) * n / (n + k),  n = maps with a recorded value."""
    decay = form_module._decay_for(HALF_LIFE)
    totals, weights, counts, rows = {}, {}, {}, []

    for (_, match_id), match in players.groupby(["played_at", "match_id"], sort=True):
        team_a, team_b = match.iloc[0]["team_a"], match.iloc[0]["team_b"]

        def side(team):
            names = match[match["Team"] == team]["Player"].unique()
            out = {}
            for column, short in STATS.items():
                known = []
                for name in names:
                    if weights.get((name, column), 0) > 0:
                        value = totals[(name, column)] / weights[(name, column)]
                        n = counts[(name, column)]
                        if k > 0:
                            value = AVERAGE_PLAYER[column] + (value - AVERAGE_PLAYER[column]) * n / (n + k)
                        known.append(value)
                out[short] = float(np.mean(known)) if known else np.nan
            return out

        form_a, form_b = side(team_a), side(team_b)
        rows.append({"match_id": match_id, **{f"{s}_diff": form_a[s] - form_b[s] for s in STATS.values()}})

        # only after reading the record does this match count towards anyone
        for _, performance in match.iterrows():
            for column in STATS:
                if pd.isna(performance[column]):
                    continue
                key = (performance["Player"], column)
                totals[key] = totals.get(key, 0.0) * decay + float(performance[column])
                weights[key] = weights.get(key, 0.0) * decay + 1.0
                counts[key] = counts.get(key, 0) + 1

    per_match = pd.DataFrame(rows).set_index("match_id")
    return learn[["match_id"]].join(per_match, on="match_id").drop(columns="match_id")


# With no pull at all, this copy must reproduce src/form.py exactly -- otherwise any
# difference below could come from the copy rather than from the shrinkage.
unshrunk = form_shrunk(0)
for column in ("rating_diff", "acs_diff"):
    assert np.allclose(unshrunk[column], form[column], equal_nan=True), column

print("copy reproduces src/form.py exactly")
print(f"average player: rating {AVERAGE_PLAYER['Rating']:.3f}, "
      f"combat score {AVERAGE_PLAYER['Average Combat Score']:.1f}")

copy reproduces src/form.py exactly
average player: rating 0.997, combat score 196.5


## 5. Results

In [6]:
def plain():
    return make_model(LogisticRegression(max_iter=5000))


def score(pipeline, frame, columns):
    """Per-fold accuracy and log loss, plus the confidence check from section 2."""
    Xf = frame[columns].astype(float)
    accuracy, losses, preds = [], [], []
    for learn_rows, check_rows in season_folds(learn):
        model = clone(pipeline).fit(Xf.loc[learn_rows], y.loc[learn_rows])
        chance = model.predict_proba(Xf.loc[check_rows])[:, 1]
        truth = y.loc[check_rows].to_numpy()
        accuracy.append(((chance >= 0.5) == truth).mean())
        losses.append(log_loss(truth, chance, labels=[0, 1]))
        preds.append(pd.DataFrame({"chance": chance, "truth": truth}))
    p = pd.concat(preds, ignore_index=True)
    confidence = np.maximum(p["chance"], 1 - p["chance"])
    right = (p["chance"] >= 0.5) == p["truth"]
    top, bottom = confidence >= confidence.quantile(2 / 3), confidence < confidence.quantile(1 / 3)
    return np.array(accuracy), float(np.mean(losses)), right[top].mean() - right[bottom].mean()


candidates = {}
for k in (0, 3, 5, 10):
    frame = pd.concat([X_agents[["map_picked_by_a"]], unshrunk if k == 0 else form_shrunk(k)], axis=1)
    name = "current model" if k == 0 else f"shrinkage, strength {k}"
    candidates[name] = score(plain(), frame, ["rating_diff", "map_picked_by_a"])
    name = "China fix" if k == 0 else f"shrinkage {k} + China fix"
    candidates[name] = score(make_final_model(), frame, FINAL_COLUMNS)

current_acc, current_loss, _ = candidates["current model"]
table = []
for name, (acc, loss, gap) in candidates.items():
    up = int((acc > current_acc + 1e-9).sum())
    down = int((acc < current_acc - 1e-9).sum())
    passes = up >= 3 and down <= 1 and loss <= current_loss + 1e-9
    table.append({
        "": name,
        "accuracy %": round(acc.mean() * 100, 1),
        "log loss": round(loss, 3),
        "per fold": "  ".join(f"{a * 100:.1f}" for a in acc),
        "vs current": "-" if name == "current model" else f"{up} better, {down} worse",
        "beats baseline": f"{int((acc > baseline['accuracy'].to_numpy()).sum())}/5",
        "confidence means something (pts)": round(gap * 100, 1),
        "keep?": "-" if name == "current model" else ("PASS" if passes else "no"),
    })

print(f"picker baseline: {baseline['accuracy'].mean() * 100:.1f}%   per fold: "
      + "  ".join(f"{a * 100:.1f}" for a in baseline["accuracy"]))
pd.DataFrame(table).set_index("")

picker baseline: 55.5%   per fold: 58.8  58.9  50.8  50.0  58.8


,accuracy %,log loss,per fold,vs current,beats baseline,confidence means something (pts),keep?
,,,,,,,
current model,59.7,0.678,61.0 56.5 57.4 59.5 64.1,-,4/5,1.9,-
China fix,61.1,0.676,61.0 60.5 58.2 59.5 66.4,"3 better, 0 worse",5/5,-0.5,PASS
"shrinkage, strength 3",59.8,0.682,58.1 58.1 57.4 63.8 61.8,"2 better, 2 worse",3/5,2.4,no
shrinkage 3 + China fix,59.8,0.676,58.1 58.1 58.2 63.8 61.1,"3 better, 2 worse",3/5,4.3,no
"shrinkage, strength 5",59.5,0.684,58.8 59.7 57.4 62.9 58.8,"2 better, 2 worse",3/5,2.4,no
shrinkage 5 + China fix,59.1,0.676,56.6 58.1 58.2 63.8 58.8,"3 better, 2 worse",2/5,6.2,no
"shrinkage, strength 10",59.1,0.688,55.1 58.9 57.4 65.5 58.8,"2 better, 2 worse",2/5,0.5,no
shrinkage 10 + China fix,59.3,0.677,54.4 58.9 59.8 63.8 59.5,"3 better, 2 worse",3/5,6.7,no


The last column is the accuracy of the model's most confident third of predictions minus its
least confident third. Near zero means its confidence tells you nothing.

### Shrinkage is dropped

It does what it was meant to do — combined with the China fix, the model's confident
predictions become up to about 7 points more reliable than its unsure ones, where before
there was hardly any difference. **But it fails the rules at every strength:** it costs
accuracy on folds 1 and 5, and on its own it makes log loss worse.

The likely reason is that early in the season *everyone* has a thin record. Pulling them all
toward the average throws away the only evidence the model has at that point. That is a
reasonable thing to revisit with more than one season of data, where early-season players
would arrive with a full year of history behind them.

### The China fix is kept

Better on 3 folds, worse on none, log loss slightly better, and it now beats the baseline on
all 5 folds instead of 4.

**This is a repair, not an accuracy gain, and it should be described that way.** The
average goes from 59.7% to 61.1%, which is 9 more maps right out of 629 — well inside the
margin of error. What it genuinely changes is coverage: on the maps it repairs, the model
stops treating a whole region as "no difference between the teams".

In [7]:
# The direct evidence: accuracy on the repaired maps themselves.
old = predictions(plain(), ["rating_diff", "map_picked_by_a"])
new = predictions(make_final_model(), FINAL_COLUMNS)

repaired = [i for i in old.index if recoverable.loc[i]]
picked = learn.loc[repaired, "map_picked_by"].map({"team_a": 1, "team_b": 0}).fillna(1)

print(f"repaired maps that the folds check: {len(repaired)}")
print(f"  current model   : {int(old.loc[repaired, 'right'].sum())} right")
print(f"  picker rule     : {int((picked == y.loc[repaired]).sum())} right")
print(f"  with China fix  : {int(new.loc[repaired, 'right'].sum())} right")

repaired maps that the folds check: 46
  current model   : 22 right
  picker rule     : 22 right
  with China fix  : 26 right


On the maps it was built to repair, the current model does no better than the picker rule —
which makes sense, since with no rating information the picker is all it has. The fix gives
it something to go on.

## 6. Does the main finding still hold for the final model?

The project's central result is that agent picks do not help. It was measured on the old
model, so it should be checked again on the new one.

In [8]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from model import RatingFallback


class FallbackKeepingAgents(RatingFallback):
    """The same repair, but passing the agent columns through instead of dropping them."""

    def transform(self, X):
        out = X.drop(columns=["acs_diff"]).copy()
        gap = out["rating_diff"].isna()
        out.loc[gap, "rating_diff"] = X.loc[gap, "acs_diff"] * self.scale_
        return out


agent_columns = [c for c in X.columns if c.startswith("agent_")]
with_agents = make_pipeline(FallbackKeepingAgents(), SimpleImputer(strategy="constant", fill_value=0.0),
                            StandardScaler(), LogisticRegression(max_iter=5000))

final_acc, final_loss, _ = candidates["China fix"]
agents_acc, agents_loss, _ = score(with_agents, X, FINAL_COLUMNS + agent_columns)

print(f"final model            : {final_acc.mean() * 100:.1f}%   log loss {final_loss:.3f}")
print(f"final model + 27 agents: {agents_acc.mean() * 100:.1f}%   log loss {agents_loss:.3f}")

final model            : 61.1%   log loss 0.676
final model + 27 agents: 54.7%   log loss 0.729


It holds. Adding the agents to the final model costs about six points and pushes log loss
above the 0.693 you get from saying "50/50" to every map.

## 7. Frozen

**Final model:** logistic regression on two columns — the difference in the ten players'
recent rating (combat score standing in where rating was not recorded) and who picked the
map. Built by `make_final_model()` in `src/model.py`, given `FINAL_COLUMNS`.

No more features from here. What remains is understanding it — error analysis and
explainability — and then scoring the held-back test data once.